[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/03_Core_Agent_Anatomy/core_agent_anatomy.ipynb)

# Module 03: Core Agent Anatomy – Deconstructing the Autonomous System

This notebook takes a build-from-scratch approach to agentic systems, using only Vanilla Python and Pydantic. No high-level agent frameworks are used.

## 1. The Body Metaphor: Agent Anatomy

We can understand an autonomous agent by analogy to the human body:

- **Brain (Model):** The reasoning engine (e.g., LLM or logic module).
- **Hands (Tools):** Functions that interact with the world (APIs, calculators, search, etc.).
- **Nervous System (Orchestration):** The control loop that coordinates reasoning and action.
- **Body/Legs (Deployment):** The environment where the agent operates (CLI, web, API, etc.).

## 2. Tools: The 'Hands' Loop

Tools are granular functions that the agent can call to interact with the world. Each tool should have a clear, single responsibility (e.g., `add`, `search_web`, not `manage_finance`).

We implement a tool-calling system from scratch using Pydantic for schema validation.

In [ ]:
from pydantic import BaseModel, Field
from typing import Any, Callable, Dict

class ToolSchema(BaseModel):
    name: str
    description: str
    args_schema: Dict[str, Any]

class Tool:
    def __init__(self, schema: ToolSchema, func: Callable):
        self.schema = schema
        self.func = func

    def invoke(self, **kwargs):
        # Validate arguments using the schema
        for arg, typ in self.schema.args_schema.items():
            if arg not in kwargs:
                raise ValueError(f"Missing argument: {arg}")
            if not isinstance(kwargs[arg], typ):
                raise TypeError(f"Argument '{arg}' must be {typ}")
        return self.func(**kwargs)

# Example: Define a simple calculator tool
calc_schema = ToolSchema(
    name="calculator",
    description="Add two numbers.",
    args_schema={"a": int, "b": int}
)
def add(a: int, b: int) -> int:
    return a + b
calculator_tool = Tool(calc_schema, add)

# Three-Part Loop: Define, Invoke, Observe
result = calculator_tool.invoke(a=2, b=3)
print(f"Calculator result: {result}")

## 3. Orchestration: The 'Nervous System'

The orchestrator is the control loop that coordinates reasoning and action. Here, we implement a simple ReAct (Reason + Act) orchestrator using a Python while loop.

In [ ]:
class AgentState(BaseModel):
    memory: list = Field(default_factory=list)
    final_answer: str = ""

# Simulated 'Brain' (LLM) for demonstration
class Brain:
    def __init__(self, tools: Dict[str, Tool]):
        self.tools = tools
        self.step = 0
    def think(self, state: AgentState):
        # For demo: alternate between math and final answer
        if self.step == 0:
            self.step += 1
            return {"thought": "I need to add two numbers.", "action": "calculator", "action_input": {"a": 5, "b": 7}}
        else:
            return {"thought": "I have the result. Done.", "action": "final_answer", "action_input": {"answer": state.memory[-1]["result"]}}

def orchestrate(brain: Brain, state: AgentState):
    while not state.final_answer:
        step = brain.think(state)
        if step["action"] == "final_answer":
            state.final_answer = step["action_input"]["answer"]
            print(f"Final Answer: {state.final_answer}")
            break
        tool = brain.tools[step["action"]]
        result = tool.invoke(**step["action_input"])
        state.memory.append({"thought": step["thought"], "action": step["action"], "result": result})
        print(f"Step: {step['thought']} | Tool: {step['action']} | Result: {result}")

# Example usage
tools = {"calculator": calculator_tool}
brain = Brain(tools)
state = AgentState()
orchestrate(brain, state)

## 4. A Taxonomy of Systems

Agentic systems can be classified by their complexity:

- **Level 0:** Core Reasoning (single agent, no tools)
- **Level 1:** Tool-using Agent (ReAct loop)
- **Level 2:** Multi-Tool Agent (multiple tools, more complex orchestration)
- **Level 3:** Collaborative Multi-Agent Systems (multiple agents, each with specialized roles)

At higher levels, we segment tasks: e.g., a "Researcher Agent" handles retrieval, a "Writer Agent" synthesizes output.

In [ ]:
# Example: Level 3 Multi-Agent Setup
class ResearcherAgent:
    def act(self, query):
        # Simulate retrieval
        return f"[RESEARCH] Found info for: {query}"

class WriterAgent:
    def act(self, context):
        # Simulate synthesis
        return f"[WRITE] Final answer based on: {context}"

# Orchestration
query = "What is the tallest mountain?"
researcher = ResearcherAgent()
writer = WriterAgent()
retrieved = researcher.act(query)
final = writer.act(retrieved)
print(final)

## 5. Hands-on Challenge: Math & Research Agent

**Challenge:**
- Build an agent that can use both a calculator tool and a mock search tool to solve a multi-step word problem (e.g., "What is the sum of the population of Paris and the result of 7 * 8?").
- Use Pydantic for all schemas.
- The orchestrator should reason, call tools, and synthesize the final answer.

*This exercise will help you understand the cyclical agentic loop and modular tool design.*

In [ ]:
# TODO: Complete the challenge below
# 1. Define a calculator tool and a mock search tool (use Pydantic schemas)
# 2. Build an agent that can use both tools to solve a multi-step problem
# 3. Orchestrate the loop and print the final answer

# Example templates:
class SearchToolSchema(ToolSchema):
    pass

def mock_search(query: str) -> str:
    # Simulate a search result
    if "population of Paris" in query:
        return "2,140,526"
    return "Unknown"

search_schema = ToolSchema(
    name="search",
    description="Search for facts.",
    args_schema={"query": str}
)
search_tool = Tool(search_schema, mock_search)

tools = {"calculator": calculator_tool, "search": search_tool}

class MathResearchBrain(Brain):
    def __init__(self, tools):
        super().__init__(tools)
        self.step = 0
    def think(self, state: AgentState):
        if self.step == 0:
            self.step += 1
            return {"thought": "I need the population of Paris.", "action": "search", "action_input": {"query": "population of Paris"}}
        elif self.step == 1:
            self.step += 1
            pop = int(state.memory[-1]["result"].replace(",", ""))
            return {"thought": "Now I need to calculate 7 * 8.", "action": "calculator", "action_input": {"a": 7, "b": 8}}
        else:
            pop = int(state.memory[0]["result"].replace(",", ""))
            product = state.memory[1]["result"]
            answer = pop + product
            return {"thought": "I have both results. Done.", "action": "final_answer", "action_input": {"answer": answer}}

# Run the orchestrator
brain = MathResearchBrain(tools)
state = AgentState()
orchestrate(brain, state)